<a href="https://colab.research.google.com/github/dkalchenko01/UA-Locations-NER-Deep-Learning/blob/validation/notebooks/02_validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Validation:

- Design a validation strategy based on common sense and expected correlation with Public leaderboard scores.
- Explain your motivation for the chosen strategy.
- Discuss possible reasons for low or high correlation with leaderboard results.
- Perform adversarial validation if necessary (make sure to do it with the same model as the main task)

### EDA insights we account for

**Data Leakage**

- All texts with the same `doc_id` must be in the same fold

- Duplicate texts must not appear in different folds

**Training Data**

- UK and ru datasets have different sizes. We plan to use mostly UK data with a smaller ru subset

- Some ru annotations are incorrect. We need to select cleaner examples

**Fold Distribution**

- Keep the number of locations per text similarly distributed across folds

- Make sure multiword locations are represented in each fold

- Keep text length distributions similar across folds

- Test texts are much longer than train texts, which may affect leaderboard scores

**Reproducibility**

- Use a fixed seed and save the folds

- All models should use the same validation split

## Environment Setup

In [1]:
import os
import ast
import re
import hashlib
import shutil
import random
from pathlib import Path
from collections import Counter, defaultdict

import sys
import base64
import subprocess

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import psutil
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import GroupKFold

In [3]:
if "google.colab" in sys.modules:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    root = "/content"
else:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    root = "/kaggle/working"

repo = "UA-Locations-NER-Deep-Learning"
branch = "validation"
repo_dir = f"{root}/{repo}"

auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
env = os.environ.copy()
env["GIT_CONFIG_COUNT"] = "1"
env["GIT_CONFIG_KEY_0"] = "http.extraheader"
env["GIT_CONFIG_VALUE_0"] = f"AUTHORIZATION: basic {auth}"

if not os.path.exists(repo_dir):
    subprocess.run(
        ["git", "clone", "-b", branch, f"https://github.com/dkalchenko01/{repo}.git", repo_dir],
        env=env, check=True
    )

print("Repository:", repo_dir)

Repository: /content/UA-Locations-NER-Deep-Learning


In [4]:
sys.path.insert(0, repo_dir)
from src.data_io import get_data_dir

ENVIRONMENT = "colab" if root == "/content" else "kaggle"

data_dir = Path(get_data_dir(ENVIRONMENT))
print("Data directory:", data_dir)

100%|██████████| 811M/811M [00:05<00:00, 162MB/s]

Extracting files...


Data directory: /root/.cache/kagglehub/competitions/kse-ua-location-extraction-2026


In [5]:
uk_df = pd.read_csv(data_dir / "uk_geo_dataset.csv")
ru_df = pd.read_csv(data_dir / "ru_geo_dataset.csv")

for name, df in [("UK", uk_df), ("ru", ru_df)]:
    print(f"{name}: {len(df):,} rows")

UK: 1,010,000 rows
ru: 8,028,840 rows


## Duplicates analysis

#### Check exact matches (symbol to symbol):

In [6]:
for name, df in [("UK", uk_df), ("ru", ru_df)]:
    duplicates = df["text"].duplicated().sum()
    print(f"{name}: {duplicates:,} duplicates ({duplicates / len(df):.2%})")

UK: 2,454 duplicates (0.24%)
ru: 523,501 duplicates (6.52%)


Check identical texts in UK and ru datasets:

In [7]:
shared = uk_df["text"].isin(ru_df["text"]).sum()
print(f"UK rows also found in ru: {shared:,} ({shared / len(uk_df):.2%})")

UK rows also found in ru: 0 (0.00%)


Check if training data contains test texts

In [23]:
test_df = pd.read_csv(data_dir / "test.csv")

for name, df in [("UK", uk_df), ("ru", ru_df)]:
  shared = df["text"].isin(test_df["text"]).sum()
  print(f"{name} rows also found in test: {shared:,}")

UK rows also found in test: 0
ru rows also found in test: 0


Checking whether identical texts have different annotations:

In [8]:
for name, df in [("UK", uk_df), ("ru", ru_df)]:
    only_repeated = df[df["text"].duplicated(keep=False)]
    conflicts = only_repeated.groupby("text")["loc_markers"].nunique().gt(1).sum()
    print(f"{name}: {conflicts:,} texts with different annotations")

UK: 0 texts with different annotations
ru: 4,786 texts with different annotations


Count all rows with conflicting annotations:

In [9]:
conflicts = ru_df.groupby("text")["loc_markers"].nunique()
conflict_texts = conflicts[conflicts > 1].index

to_be_removed = ru_df["text"].isin(conflict_texts).sum()
print(f"Texts with conflicting labels: {len(conflict_texts):,}")
print(f"Rows to remove: {to_be_removed:,} ({to_be_removed / len(ru_df):.2%})")

Texts with conflicting labels: 4,786
Rows to remove: 22,562 (0.28%)


**Decision**: Exclude ru texts with conflicting annotations

#### Similar texts, not identical copies:

Count extra duplicates after lowercasing and fixing spaces:

In [10]:
print("Ignoring register and extra whitescapes")
for name, df in [("UK", uk_df), ("ru", ru_df)]:
    normalized = df["text"].str.lower().str.split().str.join(" ")
    exact = df["text"].duplicated().sum()
    total = normalized.duplicated().sum()
    print(f"{name}: {total - exact:,} duplicates after additional normalization")

Ignoring register and extra whitescapes
UK: 65 duplicates after additional normalization
ru: 2,134 duplicates after additional normalization


**Decision**: keep only one original text with its annotations

Near duplicates in UK dataset:

In [11]:
%pip install -q datasketch

In [13]:
from datasketch import MinHash, MinHashLSH

#Split text into groups of three words
def shingles(text, k=3):
  words = re.sub(r"\W+", " ", text.lower()).split() #lowercasing, replacing non-letter symbols with whitespaces
  return {" ".join(words[i:i+k]) for i in range(max(1, len(words)-k+1))}

def minhash(text, k=3, num_perm=128):
  m = MinHash(num_perm=num_perm)
  for s in shingles(text, k):
      m.update(s.encode("utf8"))
  return m

MinHash splits texts into groups of words, hashes them in different ways and keeps the smallest hash each time.

Then it compares these minimum values: the more numbers match, the more similar the texts.


In [14]:
uk_texts = uk_df["text"].dropna().drop_duplicates()

lsh_uk = MinHashLSH(threshold=0.7, num_perm=128)
pairs_uk = set()

for i, t in uk_texts.items():
  h = minhash(t)
  for j in lsh_uk.query(h):
      pairs_uk.add((int(j), i))
  lsh_uk.insert(str(i), h)

print(f"Texts checked: {len(uk_texts):,}")
print(f"Candidate pairs: {len(pairs_uk):,}")

Texts checked: 1,007,546
Candidate pairs: 6,356


Calculate exact similarity for candidate pairs:

In [15]:
similarities_uk = []

for i, j in pairs_uk:
  words_a = shingles(uk_texts.loc[i])
  words_b = shingles(uk_texts.loc[j])
  # Jaccard similarity (intersection/union)
  shared = len(words_a & words_b)
  total = len(words_a | words_b)
  similarity = shared / total

  similarities_uk.append((i, j, similarity))

scores_uk = pd.DataFrame(similarities_uk,columns=["id1", "id2", "jaccard"])

Real examples on different thresholds:

In [16]:
for score in [0.7, 0.8, 0.9]:
  examples = scores_uk[scores_uk["jaccard"] >= score].sort_values("jaccard").head(2)
  print(f"\nJaccard >= {score}\n")
  for row in examples.itertuples():
      print(f"Similarity: {row.jaccard:.3f}")
      print("A:", uk_texts.loc[row.id1])
      print("B:", uk_texts.loc[row.id2])
      print()


Jaccard >= 0.7

Similarity: 0.700
A: Усик провів 17 поєдинків у професіоналах і здобув 17 перемог (13 нокаутом), 16 з них - у першій важкій вазі.
B: Усик провів 17 поєдинків у професіоналах і здобув 17 перемог (13 нокаутом), 16 із них - у першій важкій вазі.

Similarity: 0.700
A: Мальме - Динамо Київ у прямому ефірі дивіться на нашому сайті.
B: Шахтар Донецьк - Динамо Київ у прямому ефірі дивіться на нашому сайті.


Jaccard >= 0.8

Similarity: 0.800
A: Про це повідомляє офіційний сайт президента України.
B: Про це повідомляє офіційний сайт президента…

Similarity: 0.800
A: Центр контролю та профілактики захворювань у США, медичний інститут Національної академії наук Великобританії та британська Національна служба охорони здоров’я провели дослідження та спростували будь-який зв’язок між вакциною і виникненням аутизму.
B: Центр контролю та профілактики захворювань у США, медичний інститут Національної академії наук Великої Британії та британська Національна служба охорони здоров’я прове

In [ ]:
confirmed_uk = scores_uk[scores_uk["jaccard"] >= 0.7]

affected_uk = set(confirmed_uk["id1"]) | set(confirmed_uk["id2"])

print(f"Confirmed pairs: {len(confirmed_uk):,}")
print(f"Unique texts involved: {len(affected_uk):,}")
print(f"Share of UK unique texts: {len(affected_uk) / len(uk_texts):.2%}")

Confirmed pairs: 4,382
Unique texts involved: 8,198
Share of UK unique texts: 0.81%


#### **Conclusions**

- Found 4,382 similar (near-duplicates) UK text pairs. We keep them in the same split to avoid leakage.
- For russian data, we group sentences by doc_id.

## Preprocessing before train/val split example

Remove texts with different annotations

In [17]:
def drop_conflicting_annotations(df):
  conflicts = df.groupby("text")["loc_markers"].transform("nunique") > 1
  return df.loc[~conflicts].copy()

Keep only one copy for duplicates (ignoring case and extra spaces)

In [18]:
def drop_duplicate_texts(df):
  normalized = df["text"].str.lower().str.split().str.join(" ")
  return df.loc[~normalized.duplicated()].copy()

In [19]:
def preprocess_data(df):
  #order matters - label conflicts first, then duplicates
  df = drop_conflicting_annotations(df)
  df = drop_duplicate_texts(df)
  return df

Apply duplicates handling:

In [20]:
uk_clean = preprocess_data(uk_df)
ru_clean = preprocess_data(ru_df)

In [22]:
for name, original, clean in [("UK", uk_df, uk_clean), ("ru", ru_df, ru_clean)]:
  print(f"{name}")
  print(f"Original: {len(original):,}")
  print(f"Clean: {len(clean):,}")
  print(f"Removed: {len(original) - len(clean):,}")

UK
Original: 1,010,000
Clean: 1,007,481
Removed: 2,519
ru
Original: 8,028,840
Clean: 7,498,457
Removed: 530,383


## Validation Strategy

**Data splits**:

All models use the same saved splits, created with `GroupShuffleSplit` and seed **17**.

**Ukrainian data**
- Approximately 80% train, 10% validation and 10% holdout
- Near-duplicate texts are grouped by `near_dup_group_id` to keep similar texts in the same split

**russian data**
- Sentences are grouped by `doc_id` to keep each document in one split
- We use 100,000 training texts and ~25,000 texts each for validation and holdout
- The smaller russian training subset keeps training manageable and mostly Ukrainian, like the test set

We check text lengths, location counts and the share of texts with locations across the splits.


Validation is used to choose model settings and the best epoch. Holdout is used to tune thresholds and ensembles.

We also combine validation sentences into longer texts with line breaks and emojis. We preserve their location labels and use this set to check how models handle longer, noisy texts. It does not replace real test evaluation.


**Correlation with Leaderboard Results:**

We use the official normalized micro F1 and focus on Ukrainian scores because test is roughly 90% Ukrainian.

Local results may differ from Kaggle: our splits contain short sentences, while test contains longer posts with line breaks and emojis. Topics and annotation rules may also differ. Test has only 477 posts, so a few errors can affect the score.

### UK validation split

Groups for UK near-duplicates

In [26]:
confirmed_uk = scores_uk[scores_uk["jaccard"] >= 0.7]
remaining_pairs_uk = confirmed_uk[confirmed_uk["id1"].isin(uk_clean.index) & confirmed_uk["id2"].isin(uk_clean.index)]

parent = {i: i for i in uk_clean.index}
def find(i):
  while parent[i] != i:
      parent[i] = parent[parent[i]]
      i = parent[i]
  return i

for a, b in remaining_pairs_uk[["id1", "id2"]].itertuples(index=False, name=None):
  root_a = find(a)
  root_b = find(b)
  if root_a != root_b:
      parent[root_b] = root_a

uk_clean["near_dup_group_id"] = [find(i) for i in uk_clean.index]
print("Groups:", uk_clean["near_dup_group_id"].nunique())

Groups: 1003326


In [27]:
from sklearn.model_selection import GroupShuffleSplit

#Split into train (80%) and the remaining data (20%)
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=17)
train_idx, remaining_idx = next(splitter.split(uk_clean, groups=uk_clean["near_dup_group_id"]))
uk_train = uk_clean.iloc[train_idx].copy()
uk_remaining = uk_clean.iloc[remaining_idx]

#Split the remaining 20% into val and holdout
splitter = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=17)
val_idx, holdout_idx = next(splitter.split(uk_remaining, groups=uk_remaining["near_dup_group_id"]))
uk_val = uk_remaining.iloc[val_idx].copy()
uk_holdout = uk_remaining.iloc[holdout_idx].copy()

for name, df in [("Train", uk_train), ("Validation", uk_val), ("Holdout", uk_holdout)]:
  print(f"{name}: {len(df):,} ({len(df) / len(uk_clean):.1%})")

Train: 805,983 (80.0%)
Validation: 100,741 (10.0%)
Holdout: 100,757 (10.0%)


Location balance:

In [28]:
def count_locations(df):
    return df["loc_markers"].apply(lambda x: len(ast.literal_eval(x)) if isinstance(x, str) else len(x))

train_counts = count_locations(uk_train)
val_counts = count_locations(uk_val)
holdout_counts = count_locations(uk_holdout)

balance = pd.DataFrame({
    "Train": [(train_counts > 0).mean(), train_counts.mean()],
    "Validation": [(val_counts > 0).mean(), val_counts.mean()],
    "Holdout": [(holdout_counts > 0).mean(), holdout_counts.mean()]}, index=["Location-positive rate", "Mean locations per text"])
display(balance.round(4))

,Train,Validation,Holdout
Location-positive rate,0.2311,0.2303,0.2335
Mean locations per text,0.3416,0.3405,0.3464


Text length balance:

In [29]:
length_balance = pd.DataFrame({
    "Train": uk_train["text"].str.len().describe(percentiles=[0.5, 0.9, 0.95]),
    "Validation": uk_val["text"].str.len().describe(percentiles=[0.5, 0.9, 0.95]),
    "Holdout": uk_holdout["text"].str.len().describe(percentiles=[0.5, 0.9, 0.95])})
display(length_balance.round(1))

,Train,Validation,Holdout
count,805983.0,100741.0,100757.0
mean,102.1,102.3,102.3
std,46.6,46.6,46.7
min,39.0,40.0,40.0
50%,91.0,91.0,91.0
90%,170.0,171.0,171.0
95%,197.0,197.0,198.0
max,255.0,255.0,255.0


Location count and multiword location balance:

In [30]:
def location_stats(df):
  markers = df["loc_markers"].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
  counts = markers.apply(len)
  multiword = [any(len(text[start:end].split()) > 1 for start, end in spans) for text, spans in zip(df["text"], markers)]
  return counts.clip(upper=4).value_counts(normalize=True).sort_index(), sum(multiword) / len(df)

train_dist, train_multi = location_stats(uk_train)
val_dist, val_multi = location_stats(uk_val)
holdout_dist, holdout_multi = location_stats(uk_holdout)

distribution = pd.DataFrame({"Train": train_dist, "Validation": val_dist, "Holdout": holdout_dist}).fillna(0).rename(index={4: "4+"})
display((distribution * 100).round(2))
display(pd.DataFrame({"Train": [train_multi * 100],"Validation": [val_multi * 100], "Holdout": [holdout_multi * 100]}, index=["Texts with multiword locations (%)"]).round(2))

,Train,Validation,Holdout
loc_markers,,,
0,76.89,76.97,76.65
1,15.94,15.97,16.09
2,4.87,4.69,4.89
3,1.46,1.48,1.50
4+,0.85,0.88,0.87


,Train,Validation,Holdout
Texts with multiword locations (%),4.66,4.75,4.75


Distributions are similar, so usimg StratifiedGroupKFold would just add unnecessary complexity with no benefit.

### UK Validation manifest

Saving splits and checking for leakage

In [31]:
parts = []

for split, df in [("train", uk_train), ("val", uk_val), ("holdout", uk_holdout)]:
  part = pd.DataFrame({
      "row_id": df.index,
      "source": "uk",
      "group_id": df["near_dup_group_id"].to_numpy(),
      "split": split,
      "seed": 17})
  parts.append(part)

uk_manifest = pd.concat(parts, ignore_index=True)

if uk_manifest["row_id"].duplicated().any():
  raise ValueError("Some rows appear in multiple splits")

if len(uk_manifest) != len(uk_clean) or not uk_manifest["row_id"].isin(uk_clean.index).all():
  raise ValueError("Split rows do not match the cleaned data")

if uk_manifest.groupby("group_id")["split"].nunique().gt(1).any():
  raise ValueError("Some groups appear in multiple splits")

print(uk_manifest["split"].value_counts())

split
train      805983
holdout    100757
val        100741
Name: count, dtype: int64


### ru validation split

We use about 50,000 ru texts for validation and holdout because we only need to check how well the model works on russian data.

In [32]:
ru_splitter = GroupShuffleSplit(n_splits=1, test_size=50_000 / len(ru_clean), random_state=17)
train_idx, remaining_idx = next(ru_splitter.split(ru_clean, groups=ru_clean["doc_id"]))

ru_train = ru_clean.iloc[train_idx].sample(n=100_000, random_state=17).copy()
ru_remaining = ru_clean.iloc[remaining_idx]

# Divide the remaining data into validation and holdout.
ru_splitter = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=17)
val_idx, holdout_idx = next(ru_splitter.split(ru_remaining, groups=ru_remaining["doc_id"]))

ru_val = ru_remaining.iloc[val_idx].copy()
ru_holdout = ru_remaining.iloc[holdout_idx].copy()

Same balance checks as for UK:

In [34]:
rows = []
for name, df in [("Train", ru_train), ("Validation", ru_val), ("Holdout", ru_holdout)]:
  counts = count_locations(df)
  lengths = df["text"].str.len()
  rows.append({
      "split": name,
      "texts": len(df),
      "with locations (%)": (counts > 0).mean() * 100,
      "mean locations": counts.mean(),
      "median chars": lengths.median(),
      "p95 chars": lengths.quantile(0.95)})
pd.DataFrame(rows).set_index("split").round(2)

,texts,with locations (%),mean locations,median chars,p95 chars
split,,,,,
Train,100000,30.90,0.46,112.0,227.0
Validation,24626,31.13,0.47,113.0,229.0
Holdout,24743,31.62,0.47,113.0,228.0


All ru validation balance checks passed

### ru validation manifest

For the initial mixed-language model, we use 100,000 ru training texts to keep training manageable. We may increase this number later if time allows.

In [35]:
parts = []

for split, df in [("train", ru_train), ("val", ru_val), ("holdout", ru_holdout)]:
  part = pd.DataFrame({
      "row_id": df.index,
      "source": "ru",
      "group_id": df["doc_id"].to_numpy(),
      "split": split,
      "seed": 17})
  parts.append(part)

ru_manifest = pd.concat(parts, ignore_index=True)

if ru_manifest["row_id"].duplicated().any():
    raise ValueError("Some rows appear in multiple splits")

if ru_manifest.groupby("group_id")["split"].nunique().gt(1).any():
    raise ValueError("Some documents appear in multiple splits")

print(ru_manifest["split"].value_counts())

split
train      100000
holdout     24743
val         24626
Name: count, dtype: int64


### Combined manifest

In [36]:
validation_manifest = pd.concat([uk_manifest, ru_manifest], ignore_index=True)
pd.crosstab(validation_manifest["source"],validation_manifest["split"])[["train", "val", "holdout"]]

split,train,val,holdout
source,,,
ru,100000,24626,24743
uk,805983,100741,100757


### Test-like validation set

In [37]:
TESTLIKE_SEED = 17
TESTLIKE_SIZE = 5_000

uk_testlike_original = uk_val.sample(n=TESTLIKE_SIZE,random_state=TESTLIKE_SEED).copy()
uk_testlike_original["row_id"] = uk_testlike_original.index

print("Selected texts:", len(uk_testlike_original))
print("Unique row_id:", uk_testlike_original["row_id"].is_unique)
print("All from UK validation:", uk_testlike_original.index.isin(uk_val.index).all())
print("Texts with locations:", (count_locations(uk_testlike_original) > 0).sum())

display(uk_testlike_original[["row_id", "text", "loc_markers"]].head())

Selected texts: 5000
Unique row_id: True
All from UK validation: True
Texts with locations: 1160


,row_id,text,loc_markers
232711,232711,"В принципі, я вважаю, що ця пенсійна реформа д...",[]
598395,598395,"Звичайно, польська військова промисловість зац...","[(85, 91)]"
298924,298924,"Господарі мали територіальну перевагу, а коман...","[(51, 57)]"
740585,740585,"Як повідомляв УНІАН, 18 листопада НАК «Нафтога...",[]
66120,66120,«Сьогодні я підписую прийнятий Верховною Радою...,[]


In [38]:
rng = random.Random(17)
separators = ["\n", "\n\n", " 📍\n"]
noise = [" 🔔 #новини", " 📢", " 🔗 https://example.com"]

def parse_spans(value):
    return ast.literal_eval(value) if isinstance(value, str) else value

def make_testlike(group):
    parts, spans, row_ids = [], [], []
    offset = 0

    for _, row in group.iterrows():
        sep = rng.choice(separators) if parts else ""
        parts.append(sep + row["text"])
        offset += len(sep)

        spans.extend((start + offset, end + offset)
                     for start, end in parse_spans(row["loc_markers"]))

        offset += len(row["text"])
        row_ids.append(int(row["row_id"]))

    suffix = rng.choice(noise)
    return {
        "text": "".join(parts) + suffix,
        "loc_markers": spans,
        "source_row_ids": row_ids
    }

testlike_rows = [make_testlike(uk_testlike_original.iloc[i:i + 5]) for i in range(0, len(uk_testlike_original), 5)]
uk_testlike = pd.DataFrame(testlike_rows)

print("Test-like texts:", len(uk_testlike))
print("Original sentences used:", sum(map(len, uk_testlike["source_row_ids"])))
display(uk_testlike.head(3))

Test-like texts: 1000
Original sentences used: 5000


,text,loc_markers,source_row_ids
0,"В принципі, я вважаю, що ця пенсійна реформа д...","[(238, 244), (316, 322)]","[232711, 598395, 298924, 740585, 66120]"
1,"Своє родинне підприємство він модернізував, на...","[(386, 393)]","[476299, 528950, 360273, 192750, 723078]"
2,Відбулось розширене засідання Великої Ради ГС ...,"[(207, 218)]","[824262, 30919, 690740, 1001601, 496749]"


## Checking `labeling_sample.csv`

In [25]:
labeling_df = pd.read_csv(get_data_dir("colab") / "labeling_sample.csv")

row = labeling_df.iloc[18]
print(row["text"])
print("\nAnnotated locations:", row["locations"])

Між Києвом та Васильковом “Укрзалізниця” запустила додаткові приміські поїзди.

🚉В АТ "Укрзалізниця" призначили додаткові прискорені приміські поїзди сполученням між Васильковом та Києвом.

✔️ Курсуватимуть електрички щодня, крім вихідних, до кінця вересня.

🚊Відправлення з Приміського вокзалу Київ-Пасажирський та прибуття на станцію Васильків-Центр: 

🕒08:34 – 09:31 (поїзд №6049); 
🕑16:36 – 17:26 (поїзд №6051);
🕔18:37 – 19:27 (поїзд №6053).

🚊Відправлення зі станції Васильків-Центр та прибуття на Приміський вокзал станції Київ-Пасажирський:

🕢07:28 – 08:24 (поїзд №6050); 
🕙09:41 – 10:32 (поїзд №6052); 
🕙17:36 – 18:26 (поїзд №6054).

Вони робитимуть зупинки на станціях Боярка, Тарасівка, Вишневе, Караваєві Дачі.

Annotated locations: ['Києвом', 'Васильковом']


In `labeling_sample.csv` some locations, like Боярка and Тарасівка are not labeled.

We decided not to use this dataset for evaluation or error analysis.